# Sync RAS Curations

Syncs verb-based RAS curations (add/remove institution_ids) from the users Heroku Postgres database to a local Databricks table.

**Source**: `openalex_users.public.curations` (Heroku Postgres foreign table)
**Target**: `openalex.institutions.ras_curations` (Delta table)

Curations use verb-based semantics:
- `action='add'`: Include this institution_id even if model didn't predict it
- `action='remove'`: Exclude this institution_id even if model predicted it

Only the **latest** action per (raw_affiliation_string, institution_id) counts (oxjob #582): the curations table is an
append-only log, and a user who removes an institution and later re-adds it must end with the add. Aggregating the full
history put the same id in both arrays, and the MV's `ARRAY_EXCEPT(ARRAY_UNION(base, adds), removes)` made the stale
remove win forever.

The target table aggregates curations per RAS into arrays that can be joined into the MV.

## Sync curations from users DB

In [ ]:
%sql
-- Preview what will be synced
-- Latest action wins per (raw_affiliation_string, institution_id): a user who
-- removed an institution and later re-added it must end with the add. Without
-- this, the MV's ARRAY_EXCEPT(ARRAY_UNION(base, adds), removes) lets any
-- historical remove permanently beat a newer add (oxjob #582).
WITH latest AS (
  SELECT
    entity_id,
    action,
    CAST(REGEXP_EXTRACT(value, 'I(\\d+)', 1) AS BIGINT) AS institution_id,
    ROW_NUMBER() OVER (
      PARTITION BY entity_id, CAST(REGEXP_EXTRACT(value, 'I(\\d+)', 1) AS BIGINT)
      ORDER BY created DESC
    ) AS rn
  FROM openalex_users.public.curations
  WHERE entity = 'ras'
    AND property = 'institution_ids'
)
SELECT
  entity_id AS raw_affiliation_string,
  FILTER(
    ARRAY_AGG(CASE WHEN action = 'add' THEN institution_id END),
    x -> x IS NOT NULL
  ) AS curated_add_ids,
  FILTER(
    ARRAY_AGG(CASE WHEN action = 'remove' THEN institution_id END),
    x -> x IS NOT NULL
  ) AS curated_remove_ids,
  COUNT(*) AS num_curations
FROM latest
WHERE rn = 1
GROUP BY entity_id

In [ ]:
%sql
-- MERGE curations into local table (handles inserts, updates, AND deletes)
-- Latest action wins per (raw_affiliation_string, institution_id) — see preview
-- cell for rationale (oxjob #582).
MERGE INTO openalex.institutions.ras_curations AS target
USING (
  WITH latest AS (
    SELECT
      entity_id,
      action,
      CAST(REGEXP_EXTRACT(value, 'I(\\d+)', 1) AS BIGINT) AS institution_id,
      ROW_NUMBER() OVER (
        PARTITION BY entity_id, CAST(REGEXP_EXTRACT(value, 'I(\\d+)', 1) AS BIGINT)
        ORDER BY created DESC
      ) AS rn
    FROM openalex_users.public.curations
    WHERE entity = 'ras'
      AND property = 'institution_ids'
  )
  SELECT
    entity_id AS raw_affiliation_string,
    FILTER(
      ARRAY_AGG(CASE WHEN action = 'add' THEN institution_id END),
      x -> x IS NOT NULL
    ) AS curated_add_ids,
    FILTER(
      ARRAY_AGG(CASE WHEN action = 'remove' THEN institution_id END),
      x -> x IS NOT NULL
    ) AS curated_remove_ids,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM latest
  WHERE rn = 1
  GROUP BY entity_id
) AS source
ON target.raw_affiliation_string = source.raw_affiliation_string
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
-- Check local curations table
SELECT 
  COUNT(*) AS total_curated_ras,
  SUM(SIZE(curated_add_ids)) AS total_adds,
  SUM(SIZE(curated_remove_ids)) AS total_removes,
  MAX(updated_datetime) AS last_sync
FROM openalex.institutions.ras_curations

In [ ]:
%sql
-- Sample of curated RAS
SELECT * FROM openalex.institutions.ras_curations
ORDER BY updated_datetime DESC
LIMIT 10